In [ ]:
# ---------------------------------------------------------------------
#  Install dependencies
# ---------------------------------------------------------------------
!apt-get install -y imagemagick-6.q16
!pip install Wand
!pip install -q win2xcur
!apt-get update -qq
!apt-get install -y -qq imagemagick

# ---------------------------------------------------------------------
#  Imports and helper functions
# ---------------------------------------------------------------------
from google.colab import files
import zipfile
import os
import subprocess
import glob
import re
import shutil
import time

def sanitize_theme(name):
    return re.sub(r'[^\w\s-]', '', name).strip().replace(' ', '_')

def find_cursor_folder(base_path):
    for root, dirs, file_list in os.walk(base_path):
        if "cursors" in dirs:
            return os.path.join(root, "cursors")
        elif any('.' not in f and f.lower() not in ['index.theme', 'readme', 'license', 'copying']
                 for f in file_list):
            return root
    return None

def get_theme_name(root_path):
    for f in glob.glob(os.path.join(root_path, "**", "index.theme"), recursive=True):
        with open(f, encoding='utf-8', errors='ignore') as file:
            for line in file:
                if line.startswith("Name="):
                    return line.split("=")[1].strip()
    return os.path.basename(os.path.dirname(root_path)) if os.path.dirname(root_path) != "work" else "Custom"

def convert_one_zip(zip_path):
    base_name = os.path.splitext(os.path.basename(zip_path))[0]
    work_dir = f"work_{base_name}"
    if os.path.exists(work_dir):
        shutil.rmtree(work_dir)
    os.makedirs(work_dir, exist_ok=True)

    with zipfile.ZipFile(zip_path, 'r') as z:
        z.extractall(work_dir)

    cursor_path = find_cursor_folder(work_dir)
    if not cursor_path:
        print(f"❌ No cursor folder found in {os.path.basename(zip_path)}")
        shutil.rmtree(work_dir)
        return False

    theme = get_theme_name(work_dir)
    safe_theme = sanitize_theme(theme)

    out_dir = "output"
    os.makedirs(out_dir, exist_ok=True)
    for f in os.listdir(out_dir):
        os.remove(os.path.join(out_dir, f))

    print(f"🔄 Converting '{theme}' from {os.path.basename(zip_path)}...")

    try:
        subprocess.run(
            ['x2wincurtheme', cursor_path, '-n', theme, '-o', out_dir],
            check=True, capture_output=True
        )
    except:
        print("⚠️ Batch failed, trying individual file conversion...")
        for f in os.listdir(cursor_path):
            fp = os.path.join(cursor_path, f)
            if os.path.isfile(fp) and f not in ['index.theme', 'README', 'LICENSE']:
                subprocess.run(['x2wincur', fp, '-o', out_dir], capture_output=True)

    out_zip = f"{safe_theme}_Windows.zip"
    if os.listdir(out_dir):
        zip_abs_path = os.path.join(os.getcwd(), out_zip)
        try:
            subprocess.run(
                ['zip', '-r', zip_abs_path, '.'],
                cwd=out_dir,
                check=True,
                capture_output=True
            )
            time.sleep(1)   # ensure file is written
        except Exception as e:
            print(f"❌ Zip creation failed: {e}")
            shutil.rmtree(work_dir)
            return False

        if os.path.exists(out_zip):
            size_kb = os.path.getsize(out_zip) // 1024
            print(f"✅ Done! File: {out_zip} ({size_kb} KB)")
            print("📥 Downloading now...")
            files.download(out_zip)

            # 🔥 Wait for user to confirm the download has started
            # This gives the browser enough time to initiate the download.
            input("⏳ Press Enter after the download has started (or press Enter now if it already did)...")

            # Keep the .zip file (do not delete)
            shutil.rmtree(work_dir)   # remove temporary extraction folder
            return True
        else:
            print("❌ Zip file was not created")
    else:
        print("❌ No cursors were converted – check the zip structure")

    shutil.rmtree(work_dir, ignore_errors=True)
    return False

# ---------------------------------------------------------------------
#  Main loop – upload, process, repeat
# ---------------------------------------------------------------------
print("🖱️  Xcursor → Windows Cursor Converter")

while True:
    print("\n📤 Upload one or more Xcursor .zip files (select multiple if desired):")
    uploaded = files.upload()

    if not uploaded:
        print("No files uploaded. Exiting.")
        break

    for zip_name in uploaded.keys():
        with open(zip_name, 'wb') as f:
            f.write(uploaded[zip_name])

        convert_one_zip(zip_name)
        os.remove(zip_name)   # remove the uploaded source zip (not the output)

    again = input("\n🔄 Convert another batch? (y/n): ").strip().lower()
    if again != 'y':
        print("👋 All done!")
        # List all generated zips
        zips = [f for f in os.listdir() if f.endswith('_Windows.zip')]
        if zips:
            print("\n📦 Generated Windows cursor packs (still in the Colab runtime):")
            for z in zips:
                print(f"  - {z} ({os.path.getsize(z)//1024} KB)")
            print("You can download them again from the file browser (left panel) or use files.download().")
        break
